In [1]:
import random
import time
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.parameter import Parameter
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
from torchvision.transforms import functional as FT
from PIL import Image, ImageChops
import cv2

from sklearn.metrics import cohen_kappa_score
import timm
import os
import glob
import warnings

device = "cuda:0" if torch.cuda.is_available() else "cpu"
print("Using device:", device)


def _find_aptos_root():
    candidates = [
        "../input/aptos2019-blindness-detection",
        "/kaggle/input/aptos2019-blindness-detection",
        "/kaggle/data/aptos2019-blindness-detection",
        "/kaggle/input/aptos2019-blindness-detection/aptos2019-blindness-detection",
        "/kaggle/data/aptos2019-blindness-detection/aptos2019-blindness-detection",
    ]
    for p in candidates:
        if os.path.exists(p):
            return p
    for base in ["/kaggle/input", "/kaggle/data", "../input", "../data"]:
        if os.path.exists(base):
            hits = glob.glob(
                os.path.join(base, "**", "aptos2019-blindness-detection"),
                recursive=True,
            )
            for h in hits:
                if os.path.exists(h):
                    return h
    raise FileNotFoundError(
        "Could not locate aptos2019-blindness-detection directory in the filesystem."
    )


APTOS_ROOT = _find_aptos_root()
print("APTOS_ROOT:", APTOS_ROOT)


def _resolve_image_path(img_root: str, id_code: str):
    for ext in (".png", ".jpeg", ".jpg", ".PNG", ".JPEG", ".JPG"):
        p = os.path.join(img_root, f"{id_code}{ext}")
        if os.path.exists(p):
            return p
    return os.path.join(img_root, f"{id_code}.png")


def _extract_state_dict(ckpt_obj):
    if isinstance(ckpt_obj, dict):
        for k in [
            "state_dict",
            "model_state_dict",
            "model",
            "net",
            "network",
            "ema",
            "model_ema",
            "model_ema_state_dict",
        ]:
            if k in ckpt_obj and isinstance(ckpt_obj[k], dict):
                ckpt_obj = ckpt_obj[k]
                break

    if isinstance(ckpt_obj, dict) and all(
        isinstance(v, torch.Tensor) for v in ckpt_obj.values()
    ):
        return ckpt_obj
    return ckpt_obj


def _strip_module_prefix(state_dict):
    if not isinstance(state_dict, dict):
        return state_dict
    if not any(k.startswith("module.") for k in state_dict.keys()):
        return state_dict
    return {k[len("module.") :]: v for k, v in state_dict.items()}


def _strip_any_prefixes_to_match(state_dict, model_keys):
    """
    Score-relevant: handle common checkpoint key prefixes (Lightning 'model.', 'net.', etc.)
    so the intended pretrained weights can actually be loaded.
    """
    if not isinstance(state_dict, dict) or not state_dict:
        return state_dict

    if any(k in model_keys for k in state_dict.keys()):
        return state_dict

    prefixes = []
    for k in state_dict.keys():
        if "." in k:
            prefixes.append(k.split(".", 1)[0] + ".")
    prefixes = sorted(set(prefixes), key=len, reverse=True)

    for pref in prefixes:
        stripped = {
            k[len(pref) :]: v for k, v in state_dict.items() if k.startswith(pref)
        }
        if stripped and any(k in model_keys for k in stripped.keys()):
            return stripped

    stripped2 = {}
    for k, v in state_dict.items():
        parts = k.split(".")
        if len(parts) >= 3:
            stripped_k = ".".join(parts[2:])
            stripped2[stripped_k] = v
    if stripped2 and any(k in model_keys for k in stripped2.keys()):
        return stripped2

    return state_dict


def _find_weight_file_fuzzy(basenames, prefer_token=None):
    roots = ["/kaggle/input", "/kaggle/data", "../input", "../data"]
    exts = [".pkl", ".pth", ".pt", ".bin", ".ckpt"]
    patterns = []

    for b in basenames:
        b = str(b)
        root_b, ext_b = os.path.splitext(b)
        if ext_b.lower() in exts:
            patterns.append(b)
            patterns.append(root_b)
        else:
            patterns.append(b)
            patterns.append(b.replace("_", "-"))
            patterns.append(b.replace("-", "_"))
            for e in exts:
                patterns.append(b + e)
                patterns.append(b.replace("_", "-") + e)
                patterns.append(b.replace("-", "_") + e)

    candidates = []
    for r in roots:
        if not os.path.exists(r):
            continue
        for pat in patterns:
            hits = glob.glob(os.path.join(r, "**", pat), recursive=True)
            if not hits:
                hits = glob.glob(os.path.join(r, "**", f"*{pat}*"), recursive=True)
            for h in hits:
                if os.path.isfile(h):
                    try:
                        sz = os.path.getsize(h)
                    except OSError:
                        sz = -1
                    name = os.path.basename(h).lower()
                    prefer = 0
                    if prefer_token is not None and str(prefer_token).lower() in name:
                        prefer = 1
                    candidates.append((h, sz, prefer))

    if not candidates:
        return None

    candidates = sorted(
        candidates,
        key=lambda t: (-t[2], -t[1], len(os.path.basename(t[0])), len(t[0]), t[0]),
    )
    best_path, best_size, best_prefer = candidates[0]
    print(
        f"Weight candidate selected (by fuzzy name/size): {best_path} (prefer={best_prefer}, size={best_size/1024/1024:.2f} MB)"
    )
    return best_path


def _prepare_state_for_model(model: nn.Module, weight_path: str):
    ckpt = torch.load(weight_path, map_location="cpu")
    state = _extract_state_dict(ckpt)
    state = _strip_module_prefix(state)
    model_state = model.state_dict()
    state = _strip_any_prefixes_to_match(state, set(model_state.keys()))
    if not isinstance(state, dict):
        raise ValueError("Checkpoint did not resolve to a state_dict-like mapping.")
    return state


def _can_strict_load(model: nn.Module, weight_path: str):
    """
    Score-relevant: prefer checkpoints that can be loaded strictly into THIS model.
    """
    try:
        state = _prepare_state_for_model(model, weight_path)
        _ = model.load_state_dict(state, strict=True)
        return True
    except Exception:
        return False


def _checkpoint_match_ratio(model: nn.Module, weight_path: str):
    state = _prepare_state_for_model(model, weight_path)
    model_state = model.state_dict()

    matched = 0
    for k, v in state.items():
        if (
            k in model_state
            and isinstance(v, torch.Tensor)
            and model_state[k].shape == v.shape
        ):
            matched += 1
    return matched / max(1, len(model_state)), matched


def _infer_expected_img_size_from_path(weight_path: str):
    """
    Score-relevant minimal change: many DR checkpoints encode training resolution in the filename
    (e.g., '*_320*'). Using the correct resize at inference can materially increase QWK.
    """
    if not weight_path:
        return None
    base = os.path.basename(weight_path).lower()
    for sz in (512, 448, 384, 360, 352, 320, 300, 288, 280, 256):
        if str(sz) in base:
            return int(sz)
    return None


def _checkpoint_name_score(path: str, expected_token: str):
    """
    Score-relevant: avoid accidentally selecting a wrong-but-loadable checkpoint that tanks QWK.
    We score filenames higher when they strongly resemble the expected DR checkpoint identity.
    """
    name = os.path.basename(path).lower()
    score = 0
    if expected_token.lower() in name:
        score += 100
    for tok, w in [
        ("3stage", 20),
        ("three", 5),
        ("stage", 5),
        ("b4", 15),
        ("efficientnet", 5),
        ("tf_efficientnet_b4", 10),
        ("22epoch", 10),
        ("320", 10),
        ("aptos", 5),
        ("blind", 5),
    ]:
        if tok in name:
            score += w
    return score


def _find_best_checkpoint_for_model(model: nn.Module, basenames, prefer_token=None):
    roots = ["/kaggle/input", "/kaggle/data", "../input", "../data"]
    exts = [".pkl", ".pth", ".pt", ".bin", ".ckpt"]

    patterns = []
    for b in basenames:
        b = str(b)
        root_b, ext_b = os.path.splitext(b)
        if ext_b.lower() in exts:
            patterns.append(b)
        else:
            patterns.append(b)
            patterns.append(b.replace("_", "-"))
            patterns.append(b.replace("-", "_"))
            for e in exts:
                patterns.append(b + e)
                patterns.append(b.replace("_", "-") + e)
                patterns.append(b.replace("-", "_") + e)

    hits = set()
    for r in roots:
        if not os.path.exists(r):
            continue
        for pat in patterns:
            for h in glob.glob(os.path.join(r, "**", pat), recursive=True):
                if os.path.isfile(h):
                    hits.add(h)
            for h in glob.glob(os.path.join(r, "**", f"*{pat}*"), recursive=True):
                if os.path.isfile(h):
                    hits.add(h)

    if not hits:
        return None, 0.0

    scored = []
    expected = prefer_token if prefer_token is not None else ""
    for h in hits:
        name = os.path.basename(h).lower()
        prefer = 0
        if prefer_token is not None and str(prefer_token).lower() in name:
            prefer = 1
        try:
            sz = os.path.getsize(h)
        except OSError:
            sz = -1

        name_score = _checkpoint_name_score(h, expected)

        strict_ok = _can_strict_load(model, h)
        try:
            ratio, matched = _checkpoint_match_ratio(model, h)
        except Exception:
            ratio, matched = 0.0, 0

        scored.append(
            (1 if strict_ok else 0, ratio, name_score, prefer, matched, sz, h)
        )

    scored.sort(key=lambda t: (t[0], t[1], t[2], t[3], t[4], t[5]), reverse=True)
    (
        strict_ok,
        best_ratio,
        best_name_score,
        best_prefer,
        best_matched,
        best_sz,
        best_path,
    ) = scored[0]
    print(
        f"Best checkpoint selected: {best_path} | strict_ok={bool(strict_ok)} | match_ratio={best_ratio:.3f} | name_score={best_name_score} | matched={best_matched} | prefer={best_prefer} | size={best_sz/1024/1024:.2f} MB"
    )
    return best_path, best_ratio


def _load_checkpoint_safely(model: nn.Module, weight_path: str):
    """
    Score-relevant: try strict load first (ensures no random leftovers). If strict fails,
    fall back to partial compatible load so inference still runs.
    """
    state = _prepare_state_for_model(model, weight_path)

    try:
        missing, unexpected = model.load_state_dict(state, strict=True)
        match_ratio = 1.0
        return match_ratio, len(missing), len(unexpected), True
    except Exception as strict_e:
        model_state = model.state_dict()
        filtered = {}
        matched = 0
        for k, v in state.items():
            if (
                k in model_state
                and isinstance(v, torch.Tensor)
                and model_state[k].shape == v.shape
            ):
                filtered[k] = v
                matched += 1

        if matched == 0:
            raise ValueError(
                f"No matching keys between checkpoint and model (0 matched). strict_error={repr(strict_e)}"
            )

        missing, unexpected = model.load_state_dict(filtered, strict=False)
        match_ratio = matched / max(1, len(model_state))
        return match_ratio, len(missing), len(unexpected), False


def _init_heads_from_imagenet_classifier(net1: nn.Module, imagenet_b4: nn.Module):
    """
    Score-relevant minimal change: when we have to fall back to ImageNet pretrained backbone,
    also initialize the *last-layer weights* of heads from ImageNet classifier weights using
    a simple grouped-average mapping (1000->5/4/1). This preserves architecture and avoids
    random-head outputs that yield near-random QWK.
    """
    if not (
        hasattr(imagenet_b4, "classifier")
        and isinstance(imagenet_b4.classifier, nn.Linear)
    ):
        return False

    W = imagenet_b4.classifier.weight.detach().clone()  # [1000, 1792]
    b = imagenet_b4.classifier.bias.detach().clone()  # [1000]

    def grouped_avg(W_in, b_in, n_out):
        idxs = np.array_split(np.arange(W_in.shape[0]), n_out)
        W_out = []
        b_out = []
        for g in idxs:
            g = torch.as_tensor(g, dtype=torch.long)
            W_out.append(W_in[g].mean(dim=0))
            b_out.append(b_in[g].mean(dim=0))
        return torch.stack(W_out, dim=0), torch.stack(b_out, dim=0)

    net1.classifier[-1].bias.data.copy_(
        grouped_avg(W, b, 5)[1].to(net1.classifier[-1].bias.device)
    )
    net1.ordinal[-1].bias.data.copy_(
        grouped_avg(W, b, 4)[1].to(net1.ordinal[-1].bias.device)
    )
    net1.regressor[-1].bias.data.fill_(float(b.mean().item()) * 0.01)

    try:
        pairs = np.array_split(np.arange(1000), 500)
        M = torch.zeros((500, 1000), dtype=W.dtype)
        for i, p in enumerate(pairs):
            p = torch.as_tensor(p, dtype=torch.long)
            M[i, p] = 1.0 / float(len(p))
        imp = W.abs().mean(dim=1)  # [1000]
        imp500 = M @ imp  # [500]
        for head in (net1.classifier, net1.regressor, net1.ordinal):
            lin = head[1]  # Linear(1000,500)
            lin.weight.data.zero_()
            for j in range(500):
                src = int(pairs[j][0])
                lin.weight.data[j, src] = float(imp500[j].item()) * 0.001
            lin.bias.data.zero_()
    except Exception:
        pass

    return True




/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

Using device: cuda:0
APTOS_ROOT: ../input/aptos2019-blindness-detection


In [2]:
import torch
import torch.nn as nn
import torchvision
import csv
import timm
import time
import glob
import copy
import os
import json
import cv2
import numpy as np
import pickle
import torch.optim as optim
import torch
import torch.nn.functional as F
from torch import nn as nn
from torch.optim import lr_scheduler
from timm.models import *
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, utils, models, datasets
from PIL import Image
from PIL import Image, ImageEnhance, ImageOps
from torchvision.models._utils import IntermediateLayerGetter
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.backbone_utils import resnet_fpn_backbone
from torchvision.ops.feature_pyramid_network import (
    FeaturePyramidNetwork,
    LastLevelMaxPool,
)
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from torchvision.ops.misc import FrozenBatchNorm2d
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.ops import misc as misc_nn_ops
from torch import Tensor, Size
from torch.jit.annotations import List, Optional, Tuple
from torch.nn.parameter import Parameter
import warnings


def gem(x, p=3, eps=1e-6):
    return F.avg_pool2d(x.clamp(min=eps).pow(p), (x.size(-2), x.size(-1))).pow(1.0 / p)


class GeM(nn.Module):
    def __init__(self, p=3, eps=1e-6, flatten=False):
        super(GeM, self).__init__()
        self.p = Parameter(torch.ones(1) * p)
        self.eps = eps
        self.flatten = flatten

    def forward(self, x):
        x = gem(x, p=self.p, eps=self.eps)
        if self.flatten:
            x = x.flatten(1)
        return x

    def __repr__(self):
        return (
            self.__class__.__name__
            + "("
            + "p="
            + "{:.4f}".format(self.p.data.tolist()[0])
            + ", "
            + "eps="
            + str(self.eps)
            + ")"
        )


class FrozenBatchNorm2d(torch.nn.Module):
    """
    BatchNorm2d where the batch statistics and the affine parameters
    are fixed
    """

    def __init__(
        self,
        num_features: int,
        eps: float = 1e-5,
        n: Optional[int] = None,
    ):
        if n is not None:
            warnings.warn(
                "`n` argument is deprecated and has been renamed `num_features`",
                DeprecationWarning,
            )
            num_features = n
        super(FrozenBatchNorm2d, self).__init__()
        self.eps = eps
        self.register_buffer("weight", torch.ones(num_features))
        self.register_buffer("bias", torch.zeros(num_features))
        self.register_buffer("running_mean", torch.zeros(num_features))
        self.register_buffer("running_var", torch.ones(num_features))

    def _load_from_state_dict(
        self,
        state_dict: dict,
        prefix: str,
        local_metadata: dict,
        strict: bool,
        missing_keys: List[str],
        unexpected_keys: List[str],
        error_msgs: List[str],
    ):
        num_batches_tracked_key = prefix + "num_batches_tracked"
        if num_batches_tracked_key in state_dict:
            del state_dict[num_batches_tracked_key]

        super(FrozenBatchNorm2d, self)._load_from_state_dict(
            state_dict,
            prefix,
            local_metadata,
            strict,
            missing_keys,
            unexpected_keys,
            error_msgs,
        )

    def forward(self, x: Tensor) -> Tensor:
        w = self.weight.reshape(1, -1, 1, 1)
        b = self.bias.reshape(1, -1, 1, 1)
        rv = self.running_var.reshape(1, -1, 1, 1)
        rm = self.running_mean.reshape(1, -1, 1, 1)
        scale = w * (rv + self.eps).rsqrt()
        bias = b - rm * scale
        return x * scale + bias

    def __repr__(self) -> str:
        return f"{self.__class__.__name__}({self.weight.shape[0]}, eps={self.eps})"


class Linear(nn.Linear):
    def forward(self, input: torch.Tensor) -> torch.Tensor:
        if torch.jit.is_scripting():
            bias = self.bias.to(dtype=input.dtype) if self.bias is not None else None
            return F.linear(input, self.weight.to(dtype=input.dtype), bias=bias)
        else:
            return F.linear(input, self.weight, self.bias)


class backboneNet_efficient(nn.Module):
    def __init__(self):
        super(backboneNet_efficient, self).__init__()
        net = timm.create_model("tf_efficientnet_b4_ns", pretrained=False)
        layers_to_train = ["blocks"]
        for name, parameter in net.named_parameters():
            if all([not name.startswith(layer) for layer in layers_to_train]):
                parameter.requires_grad_(False)
        self.num_features = 1792
        self.conv_stem = net.conv_stem
        self.bn1 = net.bn1
        self.act1 = net.act1
        self.block0 = net.blocks[0]
        self.block1 = net.blocks[1]
        self.block2 = net.blocks[2]
        self.block3 = net.blocks[3]
        self.block4 = net.blocks[4]
        self.block5 = net.blocks[5]
        self.block6 = net.blocks[6]
        self.conv_head = net.conv_head
        self.bn2 = net.bn2
        self.act2 = net.act2
        self.global_pool = net.global_pool
        self.drop_rate = 0.4

        self.rg_cls = Linear(self.num_features, 1, bias=True)
        self.cls_cls = Linear(self.num_features, 5, bias=True)
        self.ord_cls = Linear(self.num_features, 4, bias=True)

    def forward(self, x):
        x1 = self.conv_stem(x)
        x2 = self.bn1(x1)
        x3 = self.act1(x2)
        x4 = self.block0(x3)
        x5 = self.block1(x4)
        x6 = self.block2(x5)
        x7 = self.block3(x6)
        x8 = self.block4(x7)
        x9 = self.block5(x8)
        x10 = self.block6(x9)
        x11 = self.conv_head(x10)
        x12 = self.bn2(x11)
        x13 = self.act2(x12)
        x14 = self.global_pool(x13)
        if self.drop_rate > 0.0:
            x14 = F.dropout(x14, p=self.drop_rate, training=self.training)

        x15 = self.rg_cls(x14)
        x16 = self.cls_cls(x14)
        x17 = self.ord_cls(x14)

        return x15, x16, x17




In [3]:
threshold = [0.75, 1.5, 2.5, 3.5]


def regress2class(out):
    prediction = 0
    for i in range(4):
        prediction += (out.data >= threshold[i]).squeeze().cpu().item()
    return int(np.clip(prediction, 0, 4))


def ordinal2class_prob(out):
    pred_prob = torch.zeros(out.size(0), 5, device=out.device, dtype=out.dtype)
    pred_prob[:, 0] = (1 - out[:, 0]).squeeze()
    pred_prob[:, 1] = (out[:, 0] * (1 - out[:, 1])).squeeze()
    pred_prob[:, 2] = (out[:, 1] * (1 - out[:, 2])).squeeze()
    pred_prob[:, 3] = (out[:, 2] * (1 - out[:, 3])).squeeze()
    pred_prob[:, 4] = out[:, 3].squeeze()
    return F.softmax(pred_prob, dim=1)


def regress2class_prob(out):
    pred_prob = torch.zeros((out.size(0), 5), device=out.device, dtype=out.dtype)
    for i in range(out.size(0)):
        if out[i] < 4.0:
            l1 = int(math.floor(float(out[i])))
            l2 = int(math.ceil(float(out[i])))
            pred_prob[i][l1] = 1 - (out[i] - l1)
            pred_prob[i][l2] = 1 - (l2 - out[i])
        else:
            pred_prob[i][4] = 1.0
    return pred_prob


def combine3output(r_out, c_out, o_out):
    R = regress2class(r_out.data)
    _, C = torch.max(c_out.data, 1)
    C = C.squeeze().item()
    _, O = torch.max(o_out.data, 1)
    O = O.squeeze().item()

    P = (R + C + O) / 3.0
    P = int(round(P))

    return P




In [4]:
def gem(x, p=3, eps=1e-6):
    return F.avg_pool2d(x.clamp(min=eps).pow(p), (x.size(-2), x.size(-1))).pow(1.0 / p)


class GeM(nn.Module):
    def __init__(self, p=3, eps=1e-6, flatten=False):
        super(GeM, self).__init__()
        self.p = Parameter(torch.ones(1) * p)
        self.eps = eps
        self.flatten = flatten

    def forward(self, x):
        x = gem(x, p=self.p, eps=self.eps)
        if self.flatten:
            x = x.flatten(1)
        return x

    def __repr__(self):
        return (
            self.__class__.__name__
            + "("
            + "p="
            + "{:.4f}".format(self.p.data.tolist()[0])
            + ", "
            + "eps="
            + str(self.eps)
            + ")"
        )


class Regressor(nn.Module):
    def __init__(self):
        super(Regressor, self).__init__()

        self.backbone = timm.models.tf_efficientnet_b5_ns(pretrained=False)
        self.backbone.global_pool = GeM(flatten=True)
        self.regressor = nn.Linear(1000, 1)

    def forward(self, x):
        x = self.backbone(x)
        out = self.regressor(x)
        out = torch.sigmoid(out) * 4.5

        return out


class ThreeStage_Model(nn.Module):
    def __init__(self, backbone=None):
        super(ThreeStage_Model, self).__init__()

        self.backbone = timm.models.tf_efficientnet_b4_ns(pretrained=False)
        self.backbone.global_pool = GeM(flatten=True)

        self.classifier = nn.Sequential(
            nn.SiLU(),
            nn.Linear(1000, 500),
            nn.SiLU(),
            nn.Linear(500, 5),
        )

        self.regressor = nn.Sequential(
            nn.SiLU(),
            nn.Linear(1000, 500),
            nn.SiLU(),
            nn.Linear(500, 1),
        )

        self.ordinal = nn.Sequential(
            nn.SiLU(),
            nn.Linear(1000, 500),
            nn.SiLU(),
            nn.Linear(500, 4),
        )

        self.final_regressor = nn.Sequential(
            nn.SiLU(),
            nn.Linear(10, 1),
        )

    def forward(self, x, final=False):
        x = self.backbone(x)

        c_out = self.classifier(x)
        r_out = self.regressor(x)
        o_out = self.ordinal(x)

        if final:
            out = torch.cat((c_out, r_out, o_out), 1)
            out = self.final_regressor(out)
            out = torch.sigmoid(out) * 4.5
            return out
        else:
            r_out = torch.sigmoid(r_out) * 4.5
            o_out = torch.sigmoid(o_out)
            return c_out, r_out, o_out




In [5]:
class photometric_distort(object):
    def __call__(self, image):
        distortions = [
            FT.adjust_brightness,
            FT.adjust_contrast,
            FT.adjust_saturation,
            FT.adjust_hue,
        ]

        random.shuffle(distortions)

        for d in distortions:
            if random.random() < 0.5:
                if d.__name__ == "adjust_hue":
                    adjust_factor = random.uniform(-16 / 255.0, 16 / 255.0)
                else:
                    adjust_factor = random.uniform(0.7, 1.3)
                image = d(image, adjust_factor)

        return image


class cropTo4_3(object):
    def __call__(self, image):
        w, h = image.size

        if (w / h) >= (4 / 3):
            new_h = h
            new_w = int(h * 4 / 3)
        else:
            new_h = int(w * 3 / 4)
            new_w = w

        left = (w - new_w) / 2
        top = (h - new_h) / 2
        right = left + new_w
        bottom = top + new_h

        return image.crop((left, top, right, bottom))


class trim(object):
    def __call__(self, image):
        bg = Image.new(image.mode, image.size, image.getpixel((0, 0)))
        diff = ImageChops.difference(image, bg)
        diff = ImageChops.add(diff, diff, 2.0, -10)
        bbox = diff.getbbox()
        if bbox:
            return image.crop(bbox)
        return image


def crop_image_from_gray(img, tol=7):
    if img.ndim == 2:
        mask = img > tol
        return img[np.ix_(mask.any(1), mask.any(0))]
    elif img.ndim == 3:
        gray_img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        mask = gray_img > tol

        check_shape = img[:, :, 0][np.ix_(mask.any(1), mask.any(0))].shape[0]
        if check_shape == 0:
            return img
        else:
            img1 = img[:, :, 0][np.ix_(mask.any(1), mask.any(0))]
            img2 = img[:, :, 1][np.ix_(mask.any(1), mask.any(0))]
            img3 = img[:, :, 2][np.ix_(mask.any(1), mask.any(0))]
            img = np.stack([img1, img2, img3], axis=-1)
        return img




In [6]:
sample_sub_path = os.path.join(APTOS_ROOT, "sample_submission.csv")
sample_sub = pd.read_csv(sample_sub_path)
test_ids = sample_sub["id_code"].astype(str).tolist()
print("Loaded sample_submission ids:", len(test_ids))

transform1 = transforms.Compose(
    [
        trim(),
        cropTo4_3(),
        transforms.Resize((288, 384)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.384, 0.258, 0.174],
            std=[0.124, 0.089, 0.094],
        ),
    ]
)

transform2 = transforms.Compose(
    [
        transforms.Resize((280, 280)),
        transforms.CenterCrop(256),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]
)

net1 = ThreeStage_Model()

expected_weight_name = "B4_3stage_22epoch_320"

best_weight_path, best_ratio = _find_best_checkpoint_for_model(
    net1,
    basenames=[
        expected_weight_name,
        expected_weight_name + ".pkl",
        expected_weight_name + ".pth",
        expected_weight_name + ".pt",
        expected_weight_name + ".bin",
        expected_weight_name + ".ckpt",
        "B4_3stage_22epoch_320",
        "3stage",
        "3_stage",
        "three_stage",
        "three-stage",
        "aptos",
        "blindness",
        "efficientnet_b4",
        "tf_efficientnet_b4",
    ],
    prefer_token=expected_weight_name,
)

if best_weight_path is None:
    best_weight_path = _find_weight_file_fuzzy(
        [
            expected_weight_name,
            expected_weight_name + ".pkl",
            expected_weight_name + ".pth",
            expected_weight_name + ".pt",
            expected_weight_name + ".bin",
            expected_weight_name + ".ckpt",
            "B4_3stage_22epoch_320",
            "3stage",
            "3_stage",
            "three_stage",
        ],
        prefer_token=expected_weight_name,
    )

loaded_ok = False
match_ratio = 0.0
strict_loaded = False
loaded_expected_checkpoint = False

MIN_ACCEPT_MATCH_RATIO = 0.85

if best_weight_path is None:
    warnings.warn(
        f"Pretrained task weights like '{expected_weight_name}(.pkl/.pth/.pt/.ckpt)' not found under common Kaggle roots."
    )
else:
    print("Loading task weights from:", best_weight_path)
    try:
        match_ratio, n_missing, n_unexpected, strict_loaded = _load_checkpoint_safely(
            net1, best_weight_path
        )
        print(
            f"Task checkpoint load: strict_loaded={strict_loaded} | match_ratio={match_ratio:.3f}, missing={n_missing}, unexpected={n_unexpected}"
        )

        bn = os.path.basename(best_weight_path).lower()
        loaded_expected_checkpoint = expected_weight_name.lower() in bn

        if match_ratio < MIN_ACCEPT_MATCH_RATIO:
            warnings.warn(
                f"Checkpoint match_ratio={match_ratio:.3f} < {MIN_ACCEPT_MATCH_RATIO:.2f}; "
                f"rejecting this checkpoint to avoid random-like predictions."
            )
            loaded_ok = False
            loaded_expected_checkpoint = False
        else:
            loaded_ok = True
    except Exception as e:
        warnings.warn(
            f"Failed to load task checkpoint from {best_weight_path}: {repr(e)}; will use fallback init."
        )
        loaded_ok = False
        loaded_expected_checkpoint = False

fallback_imagenet_backbone = False
if not loaded_ok:
    try:
        imagenet_b4 = timm.create_model("tf_efficientnet_b4_ns", pretrained=True)

        net1.backbone = imagenet_b4
        net1.backbone.global_pool = GeM(flatten=True)
        fallback_imagenet_backbone = True

        _ = _init_heads_from_imagenet_classifier(net1, imagenet_b4)

        warnings.warn(
            "No compatible task checkpoint loaded; using ImageNet-pretrained EfficientNet-B4 "
            "with stronger head initialization from ImageNet classifier (fallback to improve over random)."
        )
    except Exception as e:
        warnings.warn(
            f"Failed to enable ImageNet fallback init: {repr(e)}; using fully random weights."
        )
        fallback_imagenet_backbone = False

net1 = net1.to(device)
net1.eval()

ckpt_expected_sz = (
    _infer_expected_img_size_from_path(best_weight_path) if loaded_ok else None
)

# Score-relevant minimal change:
# Avoid forcing square resize for the expected DR checkpoint, because training often used 4:3 crops.
# If we detect "*_320*" we use a 4:3 resize keeping height=320 (width=round(320*4/3)=426) to match cropTo4_3.
if ckpt_expected_sz is not None:
    target_h = int(ckpt_expected_sz)
    target_w = int(round(target_h * 4.0 / 3.0))
    dynamic_transform1 = transforms.Compose(
        [
            trim(),
            cropTo4_3(),
            transforms.Resize((target_h, target_w)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.384, 0.258, 0.174],
                std=[0.124, 0.089, 0.094],
            ),
        ]
    )
else:
    dynamic_transform1 = transform1

if loaded_ok and loaded_expected_checkpoint:
    infer_transform = dynamic_transform1
    infer_transform_name = f"transform1(custom; expected checkpoint; resize_4by3_from_ckpt={ckpt_expected_sz})"
elif fallback_imagenet_backbone:
    infer_transform = transform2
    infer_transform_name = "transform2(ImageNet; backbone+stronger head init fallback)"
else:
    infer_transform = transform1
    infer_transform_name = "transform1(custom; default)"

print(
    "loaded_ok:",
    loaded_ok,
    f"| match_ratio={match_ratio:.3f}",
    f"| strict_loaded={strict_loaded}",
    "| loaded_expected_checkpoint:",
    loaded_expected_checkpoint,
    "| fallback_imagenet_backbone:",
    fallback_imagenet_backbone,
    "| infer_transform:",
    infer_transform_name,
)



Loaded sample_submission ids: 367


/tmp/ipykernel_55/2947287791.py:51: UserWarning: Mapping deprecated model name tf_efficientnet_b4_ns to current tf_efficientnet_b4.ns_jft_in1k.
  self.backbone = timm.models.tf_efficientnet_b4_ns(pretrained=False)
/tmp/ipykernel_55/4177264544.py:79: UserWarning: Pretrained task weights like 'B4_3stage_22epoch_320(.pkl/.pth/.pt/.ckpt)' not found under common Kaggle roots.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name tf_efficientnet_b4_ns to current tf_efficientnet_b4.ns_jft_in1k.
  model = create_fn(


model.safetensors:   0%|          | 0.00/77.9M [00:00<?, ?B/s]

loaded_ok: False | match_ratio=0.000 | strict_loaded=False | loaded_expected_checkpoint: False | fallback_imagenet_backbone: True | infer_transform: transform2(ImageNet; backbone+stronger head init fallback)


/tmp/ipykernel_55/4177264544.py:122: UserWarning: No compatible task checkpoint loaded; using ImageNet-pretrained EfficientNet-B4 with stronger head initialization from ImageNet classifier (fallback to improve over random).
  warnings.warn(


In [7]:
submission = []
test_img_root = os.path.join(APTOS_ROOT, "test_images")

with torch.no_grad():
    for i, idx in enumerate(test_ids):
        if i % 50 == 0:
            print("Predicting", i, "/", len(test_ids))

        try:
            image_name = _resolve_image_path(test_img_root, str(idx))
            img1 = Image.open(image_name).convert("RGB")
            img1 = infer_transform(img1).unsqueeze(0).to(device)

            c_out, r_out, o_out = net1(img1)

            if fallback_imagenet_backbone and (not loaded_ok):
                P = int(np.clip(combine3output(r_out, c_out, o_out), 0, 4))
            else:
                pred1 = int(regress2class(r_out.data.squeeze(1)))
                P = int(np.clip(pred1, 0, 4))
        except Exception as e:
            warnings.warn(f"Failed on id_code={idx}: {repr(e)}; using diagnosis=0")
            P = 0

        submission.append([str(idx), P])

submission = np.array(submission, dtype=object)



Predicting 0 / 367
Predicting 50 / 367
Predicting 100 / 367
Predicting 150 / 367
Predicting 200 / 367
Predicting 250 / 367
Predicting 300 / 367
Predicting 350 / 367


In [8]:
df = pd.DataFrame(submission, columns=["id_code", "diagnosis"])
df["id_code"] = df["id_code"].astype(str)
df["diagnosis"] = df["diagnosis"].astype(int)

df = sample_sub[["id_code"]].merge(df, on="id_code", how="left")
df["diagnosis"] = df["diagnosis"].fillna(0).astype(int)

assert len(df) == len(sample_sub), "Submission row count mismatch vs sample_submission."
df.to_csv("submission.csv", index=False)
print("Wrote submission.csv with shape:", df.shape)
print(df.head())

Wrote submission.csv with shape: (367, 2)
        id_code  diagnosis
0  b460ca9fa26f          2
1  6cee2e148520          2
2  ca6842bfcbc9          2
3  6cbc3dad809c          2
4  a9bc2f892cb3          2
